In [ ]:
from dotenv import find_dotenv, load_dotenv

# Searches parent folders until it finds "local.env"
load_dotenv(find_dotenv("local.env"))

# Configure Hugging Face and Flower caches to persist strictly in local data/ folder
from utils.data_processing import configure_hf_cache_dir

configure_hf_cache_dir()

In [ ]:
import tqdm as notebook_tqdm
from datasets.utils.logging import enable_progress_bar

enable_progress_bar()

## **1. Training a centralized model**

Centralized learning serves as our **theoretical performance ceiling (upper bound)**. All training data is pooled in a central repository, allowing standard globally shuffled mini-batch gradient descent.

### **A. Centralized Dataset Ingestion & Architecture Selection**

| Modality | Dataset | Input Resolution | Classes | Architecture Backbone | Description |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Digits (Low Complexity)** | `ylecun/mnist` | $28 \times 28 \times 1$ | 10 | `VGG16` (~15.0M params) | Deep convolutional feature extractor with GroupNorm. |
| **Objects (Medium Complexity)** | `uoft-cs/cifar10` | $32 \times 32 \times 3$ | 10 | `ResNet50` (~23.5M params) | 50-layer deep bottleneck residual network with GroupNorm. |

---


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax

from models import count_parameters, get_model, list_models
from models.resnet import ResNet50
from models.vgg import VGG16, VGG19
from utils.augmentation import augment_cifar10_batch, normalize_cifar10
from utils.data_processing import load_processed_dataset

In [ ]:
def load_centralized_arrays(dataset_name, max_train=5000, max_test=1000, seed=42):
    """Load and normalize centralized train and test splits into JAX arrays."""
    train_ds = load_processed_dataset(dataset_name, split="train")
    test_ds = load_processed_dataset(dataset_name, split="test")

    img_key = "img" if "img" in train_ds.column_names else "image"
    is_cifar = "cifar" in dataset_name.lower()

    def to_np(ds, limit):
        sub = ds.select(range(min(len(ds), limit)))
        imgs = [
            np.expand_dims(np.array(im, dtype=np.float32) / 255.0, -1)
            if np.array(im).ndim == 2
            else np.array(im, dtype=np.float32) / 255.0
            for im in sub[img_key]
        ]
        imgs_arr = np.stack(imgs)
        if is_cifar:
            imgs_arr = normalize_cifar10(imgs_arr)
        return jnp.array(imgs_arr), jnp.array(np.array(sub["label"], dtype=np.int32))

    x_train, y_train = to_np(train_ds, max_train)
    x_test, y_test = to_np(test_ds, max_test)
    return (x_train, y_train), (x_test, y_test)

In [ ]:
# JIT-compiled optimization step
@jax.jit
def train_step(state, batch_x, batch_y):
    def loss_fn(params):
        logits = state.apply_fn({"params": params}, batch_x, train=True)
        loss = optax.softmax_cross_entropy_with_integer_labels(
            logits=logits, labels=batch_y
        )
        return jnp.mean(loss)

    grad_fn = jax.grad(loss_fn)
    grads = grad_fn(state.params)
    return state.apply_gradients(grads=grads)

In [ ]:
def make_eval_step(apply_fn):
    """Construct a JIT-compiled evaluation function for arbitrary model backbones."""

    @jax.jit
    def eval_step(params, batch_x, batch_y):
        logits = apply_fn({"params": params}, batch_x, train=False)
        loss = jnp.mean(
            optax.softmax_cross_entropy_with_integer_labels(
                logits=logits, labels=batch_y
            )
        )
        preds = jnp.argmax(logits, axis=-1)
        acc = jnp.mean(preds == batch_y)
        return loss, acc

    return eval_step

In [ ]:
def train_centralized(dataset_name, epochs=10, batch_size=64, lr=1e-3, model=None):
    """Execute centralized training loop on pooled data with dataset-tailored architecture.

    - MNIST: Uses VGG-16 / VGG-19 with GroupNorm for deep hierarchical feature extraction.
    - CIFAR-10: Uses ResNet-50 (23.5M params) with 3-layer Bottleneck residual blocks,
      GroupNorm, and data augmentation (random horizontal flip + crop).
    """
    (x_train, y_train), (x_test, y_test) = load_centralized_arrays(dataset_name)
    n_train = len(x_train)
    is_cifar = "cifar" in dataset_name.lower()

    if model is None:
        model = (
            get_model("resnet50", num_classes=10)
            if is_cifar
            else get_model("vgg16", num_classes=10)
        )

    eval_fn = make_eval_step(model.apply)

    key = jax.random.PRNGKey(42)
    init_params = model.init(key, x_train[:2], train=False)["params"]
    state = train_state.TrainState.create(
        apply_fn=model.apply,
        params=init_params,
        tx=optax.adamw(lr),
    )

    history = {"epoch": [], "loss": [], "acc": []}
    rng = np.random.default_rng(42)

    for epoch in range(1, epochs + 1):
        perm = rng.permutation(n_train)
        for s in range(0, n_train, batch_size):
            idx = perm[s : s + batch_size]
            bx, by = np.array(x_train[idx]), y_train[idx]
            if is_cifar:
                bx = augment_cifar10_batch(bx, rng=rng)
            state = train_step(state, jnp.array(bx), by)

        loss, acc = eval_fn(state.params, x_test, y_test)
        history["epoch"].append(epoch)
        history["loss"].append(float(loss))
        history["acc"].append(float(acc) * 100.0)
        print(
            f"  Epoch {epoch:02d} -> Test Acc: {float(acc) * 100:5.2f}% | Loss: {float(loss):.4f}"
        )

    return history, state.params

In [ ]:
from flax.training import train_state

centralized_results = {}
for name in ["ylecun/mnist", "uoft-cs/cifar10"]:
    tag = "MNIST" if "mnist" in name else "CIFAR-10"
    arch_name = "VGG-16" if tag == "MNIST" else "ResNet-50"
    print(f"\n[*] Training Centralized Model ({arch_name}) on {tag}...")
    hist, params = train_centralized(name, epochs=10)
    centralized_results[tag] = {"history": hist, "params": params, "model": arch_name}

### **B. Centralized Convergence Visualizations**


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Accuracy Curves
for tag, data in centralized_results.items():
    h = data["history"]
    axes[0].plot(h["epoch"], h["acc"], marker="o", linewidth=2, label=f"{tag}")
axes[0].set_title("Centralized Test Accuracy vs Epochs", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Test Accuracy (%)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.6)

# Loss Curves
for tag, data in centralized_results.items():
    h = data["history"]
    axes[1].plot(h["epoch"], h["loss"], marker="s", linewidth=2, label=f"{tag}")
axes[1].set_title("Centralized Test Loss vs Epochs", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Test Loss")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

## **2. Training a federated model**

Now we simulate decentralized learning across $K = 5$ edge clients under an **ideal IID partition** using `IidPartitioner`.

### **A. Federated Partitions Setup**


In [ ]:
from flwr_datasets.partitioner import IidPartitioner
from flwr_datasets.visualization import plot_label_distributions

from utils.data_processing import get_federated_dataset

In [ ]:
num_clients = 5

fds_fed = {
    "MNIST": get_federated_dataset(
        "mnist",
        partitioners={"train": IidPartitioner(num_partitions=num_clients)},
    ),
    "CIFAR-10": get_federated_dataset(
        "cifar10",
        partitioners={"train": IidPartitioner(num_partitions=num_clients)},
    ),
}

In [ ]:
# Visualizing partition distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for idx, (tag, fds) in enumerate(fds_fed.items()):
    plot_label_distributions(
        partitioner=fds.partitioners["train"],
        label_name="label",
        plot_type="bar",
        size_unit="percent",
        axis=axes[idx],
        title=f"{tag}: Federated IID Partitions (5 Clients)",
        legend=(idx == 1),
    )

plt.tight_layout()
plt.show()

In [ ]:
from federated import (
    BaseAggregator,
    FedAdam,
    FedAvg,
    FedMedian,
    FedProx,
    FedTrimmedMean,
    get_aggregator,
    list_aggregators,
    train_federated,
)

print(f"[✓] Loaded plug-and-play federated aggregators: {list_aggregators()}")

In [ ]:
federated_results = {}
for tag, fds in fds_fed.items():
    hub_name = "ylecun/mnist" if tag == "MNIST" else "uoft-cs/cifar10"
    fed_model = (
        get_model("vgg16", num_classes=10)
        if tag == "MNIST"
        else get_model("resnet50", num_classes=10)
    )
    arch_name = fed_model.__class__.__name__
    print(f"\n[*] Training Federated Model (FedAvg with {arch_name}) on {tag}...")
    hist, params = train_federated(
        fds, hub_name, aggregator="fedavg", rounds=10, model=fed_model
    )
    federated_results[tag] = {"history": hist, "params": params, "model": arch_name}

### **B. Federated Convergence Visualizations**


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Accuracy Curves
for tag, data in federated_results.items():
    h = data["history"]
    axes[0].plot(h["round"], h["acc"], marker="o", linewidth=2, label=f"{tag}")
axes[0].set_title("Federated Test Accuracy vs Communication Rounds", fontweight="bold")
axes[0].set_xlabel("Communication Round")
axes[0].set_ylabel("Test Accuracy (%)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.6)

# Loss Curves
for tag, data in federated_results.items():
    h = data["history"]
    axes[1].plot(h["round"], h["loss"], marker="s", linewidth=2, label=f"{tag}")
axes[1].set_title("Federated Test Loss vs Communication Rounds", fontweight="bold")
axes[1].set_xlabel("Communication Round")
axes[1].set_ylabel("Test Loss")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

## **3. Comparison of Key performance metrics**

### **A. Metrics Summary Table**


In [ ]:
import pandas as pd

rows = []
for tag in ["MNIST", "CIFAR-10"]:
    c_acc = centralized_results[tag]["history"]["acc"][-1]
    c_loss = centralized_results[tag]["history"]["loss"][-1]
    f_acc = federated_results[tag]["history"]["acc"][-1]
    f_loss = federated_results[tag]["history"]["loss"][-1]
    model_name = centralized_results[tag].get("model", "CNN")
    gap = c_acc - f_acc

    rows.append(
        {
            "Dataset": tag,
            "Backbone Architecture": model_name,
            "Centralized Acc (%)": round(c_acc, 2),
            "Centralized Loss": round(c_loss, 4),
            "Federated Acc (%)": round(f_acc, 2),
            "Federated Loss": round(f_loss, 4),
            "Decentralization Gap (%p)": round(gap, 2),
        }
    )

metrics_df = pd.DataFrame(rows)
metrics_df

### **B. Overlay Trajectory Comparison**

Direct comparison of Centralized SGD (over epochs) vs Federated Averaging (over communication rounds):


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for idx, tag in enumerate(["MNIST", "CIFAR-10"]):
    c_h = centralized_results[tag]["history"]
    f_h = federated_results[tag]["history"]

    axes[idx].plot(
        c_h["epoch"],
        c_h["acc"],
        marker="o",
        linewidth=2.5,
        label="Centralized (Upper Bound)",
        color="#1f77b4",
    )
    axes[idx].plot(
        f_h["round"],
        f_h["acc"],
        marker="s",
        linewidth=2.5,
        linestyle="--",
        label="Federated IID (FedAvg)",
        color="#ff7f0e",
    )
    axes[idx].set_title(f"{tag}: Centralized vs Federated Accuracy", fontweight="bold")
    axes[idx].set_xlabel("Iteration Step (Epoch / Communication Round)")
    axes[idx].set_ylabel("Global Test Accuracy (%)")
    axes[idx].legend(frameon=True, facecolor="white")
    axes[idx].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

### **C. Decentralization Penalty by Modality**


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

bars = ax.bar(
    metrics_df["Dataset"],
    metrics_df["Decentralization Gap (%p)"],
    color=["#3498db", "#e67e22"],
    edgecolor="black",
    width=0.45,
)

In [ ]:
for bar in bars:
    yval = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        yval + 0.3,
        f"{yval:+.2f}%p",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax.set_title(
    "Decentralization Gap ($Acc_{Centralized} - Acc_{Fed-IID}$)", fontweight="bold"
)
ax.set_ylabel("Accuracy Penalty (% points)")
ax.set_ylim(0, max(metrics_df["Decentralization Gap (%p)"]) + 3)
ax.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

### **D. Key Insights & Takeaways**

* **Privacy vs Performance Trade-off:** Federated learning preserves data locality across clients at the cost of a modest decentralization penalty compared to pooled centralized SGD.
* **Modality Robustness:** On structured datasets like MNIST, Federated IID closely matches Centralized accuracy with negligible degradation. On complex RGB datasets like CIFAR-10, the gap is wider due to periodic parameter averaging instead of continuous gradient updates.
* **Next Step:** Having established the Centralized Upper Bound and Federated IID baseline, notebook `02-iid-vs-noniid.ipynb` investigates how **statistical data heterogeneity (Non-IID skew)** further impacts this performance gap.


## **4. Plug-and-Play Federated Aggregation Comparison**

Federated aggregation algorithms dictate how edge model updates are synthesized into a global model.
We compare 5 plug-and-play strategies on MNIST:

1. **FedAvg (Federated Averaging):** Canonical sample-weighted coordinate averaging (McMahan et al., 2017).
2. **FedProx:** Adds proximal regularization ($\mu = 0.01$) to penalize client parameter drift (Li et al., 2020).
3. **FedMedian:** Coordinate-wise median, inherently robust to outlier or poisoned client updates (Yin et al., 2018).
4. **FedTrimmedMean:** Discards extreme coordinate values ($\beta = 0.1$) before averaging (Yin et al., 2018).
5. **FedAdam:** Server-side adaptive moment optimization over pseudo-gradients (Reddi et al., 2020).


In [ ]:
# Define plug-and-play aggregation strategies to evaluate
agg_strategies = {
    "FedAvg": FedAvg(),
    "FedProx (μ=0.01)": FedProx(mu=0.01),
    "FedMedian": FedMedian(),
    "FedTrimmedMean (β=0.1)": FedTrimmedMean(beta=0.1),
    "FedAdam (η=0.05)": FedAdam(eta=0.05),
}

vgg_mnist_model = get_model("vgg16", num_classes=10)

agg_benchmark_results = {}
for name, strat in agg_strategies.items():
    print(f"\n[*] Training with {name} on MNIST (VGG-16)...")
    hist, params = train_federated(
        fds_fed["MNIST"],
        "ylecun/mnist",
        aggregator=strat,
        rounds=5,
        num_clients=num_clients,
        model=vgg_mnist_model,
    )
    agg_benchmark_results[name] = {"history": hist, "params": params}

In [ ]:
# Visualizing convergence across all 5 plug-and-play aggregation strategies
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

palette = ["#2ecc71", "#3498db", "#9b59b6", "#e67e22", "#e74c3c"]
markers = ["o", "s", "^", "D", "v"]

# 1. Test Accuracy Curves
for (name, data), color, marker in zip(
    agg_benchmark_results.items(), palette, markers, strict=True
):
    h = data["history"]
    axes[0].plot(
        h["round"], h["acc"], marker=marker, linewidth=2, label=name, color=color
    )
axes[0].set_title("Aggregator Benchmark: Test Accuracy vs Rounds", fontweight="bold")
axes[0].set_xlabel("Communication Round")
axes[0].set_ylabel("Test Accuracy (%)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.6)

# 2. Test Loss Curves
for (name, data), color, marker in zip(
    agg_benchmark_results.items(), palette, markers, strict=True
):
    h = data["history"]
    axes[1].plot(
        h["round"], h["loss"], marker=marker, linewidth=2, label=name, color=color
    )
axes[1].set_title("Aggregator Benchmark: Test Loss vs Rounds", fontweight="bold")
axes[1].set_xlabel("Communication Round")
axes[1].set_ylabel("Test Loss")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Tabular comparison of plug-and-play aggregation techniques
agg_summary_rows = []
for name, data in agg_benchmark_results.items():
    h = data["history"]
    agg_summary_rows.append(
        {
            "Aggregation Strategy": name,
            "Final Test Acc (%)": round(h["acc"][-1], 2),
            "Final Test Loss": round(h["loss"][-1], 4),
            "Peak Test Acc (%)": round(max(h["acc"]), 2),
        }
    )

agg_summary_df = pd.DataFrame(agg_summary_rows)
agg_summary_df